# Runtime SDE semantics
Pinned scdiffeq 1.1.4 / neural-diffeqs 0.4.1. A potential network emits a scalar; teacher drift is its state gradient, not the raw scalar network output. G has one Brownian column and is not independent noise for each PC.

In [1]:
from pathlib import Path
import os,sys,json
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
os.chdir(ROOT);sys.path.insert(0,str(ROOT))


In [2]:
from src.teacher import TeacherSDE
import numpy as np,torch,inspect
teacher=TeacherSDE(Path('outputs/baseline/official/teacher.ckpt'))
x=np.load('outputs/distillation/observed_states.npy')[:8]
f,g=teacher.evaluate(x)
tensor=torch.as_tensor(x,dtype=torch.float32)
phi=teacher.sde.mu(tensor)
sigma=teacher.sde.sigma(tensor)
D=g@g.transpose(0,2,1)
print(type(teacher.lightning_model));print(teacher.sde)
print('state',x.shape,'potential NN',tuple(phi.shape),'sigma NN',tuple(sigma.shape))
print('f',f.shape,'G',g.shape,'D',D.shape)
print('sde_type',teacher.sde_type,'noise_type',teacher.noise_type,'brownian_dim',teacher.brownian_dim)
print('min D eigenvalue',np.linalg.eigvalsh(D.astype('float64')).min())

<class 'scdiffeq.core.lightning_models._lightning_sde_fixed_potential_regularized_velocity_ratio.LightningSDE_FixedPotential_RegularizedVelocityRatio'>
PotentialSDE(
  (mu): ABCParse.ABCParse
  (sigma): ABCParse.ABCParse
)
state (8, 50) potential NN (8, 1) sigma NN (8, 50)
f (8, 50) G (8, 50, 1) D (8, 50, 50)
sde_type ito noise_type general brownian_dim 1
min D eigenvalue -2.1641040360721327e-08


In [3]:
print(inspect.getsource(teacher.sde.f));print(inspect.getsource(teacher.sde.g))
print(inspect.getsource(teacher.sde.drift))

    def f(self, t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """Compute the drift term of the differential equation.
        
        This method provides a standardized interface for the drift term,
        compatible with numerical solvers like those in torchdiffeq and torchsde.
        
        Args:
            t (torch.Tensor): The current time point.
            y (torch.Tensor): The current state of the system.
            
        Returns:
            torch.Tensor: The drift term evaluated at the current state.
        """
        return self.drift(y)

    def g(self, t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """Compute the diffusion term of the differential equation.
        
        This method provides a standardized interface for the diffusion term,
        compatible with numerical solvers like those in torchsde.
        
        Args:
            t (torch.Tensor): The current time point.
            y (torch.Tensor): The current state of the

Installed f/g source ignores explicit time and forwards state only. The potential drift requires autograd even during evaluation. D=GGᵀ is the Itô infinitesimal covariance (not a finite-time covariance), PSD and rank at most one here. Symbolic G can preserve this factorized PSD structure. Diffusion magnitude shown in plotting helpers is not the full G tensor. Public field correspondence was verified in the baseline run.

In [4]:
sem=json.loads(Path('outputs/baseline/official/semantics.json').read_text())
sem.update(potential_network_output_shape=list(phi.shape),diffusion_network_raw_output_shape=list(sigma.shape),drift_definition='coef_drift * gradient_x potential(x)',drift_parameters=sum(p.numel() for p in teacher.sde.mu.parameters()),diffusion_parameters=sum(p.numel() for p in teacher.sde.sigma.parameters()))
Path('outputs/logs/teacher_semantics_extended.json').write_text(json.dumps(sem,indent=2))
{k:sem[k] for k in ('public_api_matches_direct','drift_parameters','diffusion_parameters','potential_network_output_shape')}

{'public_api_matches_direct': True,
 'drift_parameters': 289280,
 'diffusion_parameters': 4338,
 'potential_network_output_shape': [8, 1]}